[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/3-agent-loop.ipynb)

## Colab setup

Run the next cell first on Google Colab. It sparse-checkouts this public repo and installs packages only when the kernel is Colab. Local Jupyter and VS Code skip that work.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) or in an environment cell: `os.environ["OPENAI_API_KEY"] = "sk-..."`. Do that before the lesson setup cell. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

In [ ]:
# Colab setup. Run this cell before the other code cells.
# A fresh Colab runtime cannot import tutorial.common yet, so this cell
# sparse-checkouts the repo first. Local Jupyter and VS Code skip that.
#
# An API key is required. On Google Colab, set OPENAI_API_KEY in one of these ways:
#   * Secrets (the key icon): a secret named OPENAI_API_KEY
#   * an environment cell: os.environ["OPENAI_API_KEY"] = "sk-..."
# Locally, put the key in the repository-root .env.
# Do not commit a key. This cell does not print the value.
# Edits you make in Colab stay in the session. They do not push to GitHub.

import sys
from pathlib import Path


def _on_colab():
    try:
        import google.colab
    except ImportError:
        return False
    return Path("/content").is_dir() and google.colab is not None


def _use(root):
    if not (Path(root) / "tutorial" / "common" / "colab.py").is_file():
        return False
    text = str(root)
    if text not in sys.path:
        sys.path.insert(0, text)
    return True


ready = False
if _on_colab():
    repo = Path("/content/book-agents")
    if not _use(repo):
        import shutil
        import subprocess

        if repo.exists() and not (repo / ".git").exists():
            raise RuntimeError(
                str(repo) + " exists but is not a git checkout. "
                "Move that folder aside and run this cell again."
            )
        if repo.exists():
            shutil.rmtree(repo)
        url = "https://github.com/junlinghu/book-agents.git"
        try:
            subprocess.check_call([
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                url, str(repo),
            ])
            subprocess.check_call([
                "git", "-C", str(repo), "sparse-checkout", "set", "tutorial",
            ])
        except subprocess.CalledProcessError:
            if repo.exists():
                shutil.rmtree(repo)
            subprocess.check_call(["git", "clone", "--depth", "1", url, str(repo)])
        if not _use(repo):
            raise RuntimeError(
                "Colab setup could not find tutorial/common/colab.py after cloning."
            )
    ready = True
else:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if _use(candidate):
            ready = True
            break
    if not ready:
        print("Not Colab. Skipped clone and pip install.")

if ready:
    from tutorial.common.colab import setup_colab

    setup_colab()


# 3. The agent loop

Tutorials 1 and 2 ran one exchange by hand: model, tools, model. A real question can need several of those exchanges.

`run_agent` is that exchange inside a `for` loop. It stops when the model answers, when the step cap hits, or when the same call repeats. The shelf tool and the file tool from tutorial 2 are imported above. This notebook adds the loop.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, the setup cell above reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `3-agent-loop.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '3-agent-loop'
LESSON_NUMBER = 3
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
from tutorial.common.harness import check, system_text
from tutorial.common.shelf import reset_db


## Tutorial 3 — the agent loop

One step is one model call. Tool results are the observation. The loop stops on a final answer, on `max_steps`, or on a repeated call. It does not write a customer answer when it stops early. Tutorials 10, 13, and 14 register a gate, spans, and a cache on `HOOKS`. Those stay off in this notebook.

In [ ]:
"""Agent loop for tutorials 3 through 15.

One step is one model call. The loop stops on a final answer, on
``max_steps``, or when the same tool call repeats.

Later lessons register optional pieces on ``HOOKS``: a gate (tutorial 10),
trace spans (tutorial 13), and a token ledger with a read cache (tutorial 14).
Those stay off until the lesson imports them.
"""

import json

from tutorial.common.harness import (
    HOOKS,
    TOOLS,
    assistant_message,
    call_tool,
    preview,
)
from tutorial.runtime import chat

MAX_IDENTICAL_CALLS = 2
REPEAT_NOTE = (
    "\n\nNOTE: You already made this call. Answer the question without repeating it."
)


def run_agent(user_text, system, max_steps=6, confirmed_tokens=None, trace_id="tutorial"):
    """Perceive the thread, let the model reason, act on tool calls, observe the results."""
    gate_call = HOOKS.get("gate_call")
    format_decision = HOOKS.get("format_decision")
    root_span = HOOKS.get("root_span")
    make_span = HOOKS.get("make_span")
    route_task = HOOKS.get("route_task")
    costing = HOOKS.get("cached_call") is not None
    confirmed = set(confirmed_tokens or [])
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_text},
    ]
    seen = {}
    log = []
    spans = []
    usage_rows = []
    if root_span:
        spans.append(root_span(trace_id))
    for step in range(1, max_steps + 1):
        if route_task and step == 1:
            routed = route_task(user_text)
            print(
                "route: task=" + routed["task"]
                + " model=" + routed["model"]
                + " (" + routed["reason"] + ")"
            )
        turned = chat(messages, TOOLS)
        if costing:
            usage_rows.append({
                "step": step,
                "prompt_tokens": turned["usage"]["prompt_tokens"],
                "completion_tokens": turned["usage"]["completion_tokens"],
                "latency_ms": turned["latency_ms"],
            })
            print(
                "usage: step=" + str(step)
                + " prompt_tokens=" + str(turned["usage"]["prompt_tokens"])
                + " completion_tokens=" + str(turned["usage"]["completion_tokens"])
                + " latency_ms=" + str(turned["latency_ms"])
            )
        messages.append(assistant_message(turned))
        calls = turned["tool_calls"] or []
        if not calls:
            text = (turned["content"] or "").strip() or "(empty answer)"
            print("stop: final after " + str(step) + " model call(s)")
            return _finish(text, step, "final", log, spans, usage_rows)
        for call in calls:
            signature = (
                call["name"] + " "
                + json.dumps(call["arguments"], sort_keys=True, default=str)
            )
            count = seen.get(signature, 0) + 1
            seen[signature] = count
            if count > MAX_IDENTICAL_CALLS:
                log.append("step " + str(step) + ": repeated " + signature)
                print("stop: repeated_call after " + str(step) + " model call(s)")
                text = (
                    "Stopped: the model repeated the same tool call ("
                    + call["name"] + ") more than " + str(MAX_IDENTICAL_CALLS)
                    + " times. The harness ended the loop."
                )
                return _finish(text, step, "repeated_call", log, spans, usage_rows)
            if gate_call:
                decision = gate_call(call["name"], call["arguments"], confirmed)
                print(format_decision(decision))
                if decision["decision"] != "allowed":
                    result = decision["decision"].upper() + ": " + decision["reason"]
                    if decision["tier"] == "confirm":
                        result += " token=" + decision["approval_token"]
                else:
                    result = _invoke(call)
            else:
                result = _invoke(call)
            if count == MAX_IDENTICAL_CALLS:
                result += REPEAT_NOTE
            first = result.splitlines()[0] if result else "(empty)"
            log.append(
                "step " + str(step) + ": " + call["name"] + " "
                + json.dumps(call["arguments"], sort_keys=True, default=str)
                + " -> " + first
            )
            if make_span:
                spans.append(make_span(trace_id, step, call["name"], result))
            messages.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "content": result,
            })
            print(
                "[step " + str(step) + "] " + call["name"] + " "
                + json.dumps(call["arguments"], sort_keys=True)
            )
            print(preview(result))
            print()
    text = (
        "Stopped: reached max steps (" + str(max_steps)
        + ") without a final answer. The harness did not write one."
    )
    print("stop: max_steps after " + str(max_steps) + " model call(s)")
    return _finish(text, max_steps, "max_steps", log, spans, usage_rows)


def _invoke(call):
    cached_call = HOOKS.get("cached_call")
    if cached_call:
        result, _hit = cached_call(call)
        return result
    return call_tool(call)


def _finish(text, steps, stopped, log, spans, usage_rows):
    return {
        "text": text,
        "steps": steps,
        "stopped": stopped,
        "tool_log": log,
        "spans": spans,
        "usage": usage_rows,
    }


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
reset_db()
question = "Which products are at or below their reorder point, and may we ship oat milk?"

print("=== perceive, reason, act, observe ===")
result = run_agent(question, system_text())
print("ANSWER:", result["text"])
check(result["stopped"] == "final", "loop stopped with a final answer")
check(result["steps"] >= 2, "the loop used more than one model call")
check(any("query_inventory" in line for line in result["tool_log"]), "inventory ran inside the loop")
check(any("read_shop_file" in line for line in result["tool_log"]), "policy was read on a later step")
check("does not ship" in result["text"].lower(), "the answer uses the file, not a guess")

print("=== max steps ===")
capped = run_agent(question, system_text(), max_steps=1)
print("ANSWER:", capped["text"])
check(capped["stopped"] == "max_steps", "max_steps stops without inventing an answer")
check("Stopped: reached max steps" in capped["text"], "the harness does not write a customer answer")

print("=== repeated call ===")
repeated = run_agent("List low stock again.", system_text(), max_steps=6)
print("ANSWER:", repeated["text"])
check(repeated["stopped"] == "repeated_call", "the third identical call stops the loop")
